# Soccer ViT Line-Break (Colab GPU)

이 노트북은 `soccer-vit-linebreak` 프로젝트를 Colab GPU에서 빠르게 실행하기 위한 스모크/중간실험용 워크플로우입니다.

- Metrica sample-data clone
- dataset build
- baseline / ViT smoke training
- eval / report / explainability artifact 확인


In [ ]:
!nvidia-smi || true
import os, platform
print(platform.platform())


## 1) 프로젝트 가져오기

아래 `REPO_URL`를 본인 레포 URL로 바꾸세요. (GitHub private이면 토큰/SSH 필요)


In [ ]:
REPO_URL = 'https://github.com/<your-org>/<your-repo>.git'
REPO_DIR = '/content/soccer-vit-linebreak'
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
%cd /content/soccer-vit-linebreak


## 2) 패키지 설치

Colab의 torch가 이미 설치된 경우가 많지만 버전 충돌이 있으면 아래에서 재설치하세요.


In [ ]:
%cd /content/soccer-vit-linebreak
!python -m pip install -U pip
!python -m pip install pyyaml scikit-learn matplotlib timm
# 필요시 torch/torchvision 재설치 (주석 해제)
# !python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!python - <<'PY'
import torch, timm
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
print('timm', timm.__version__)
PY


## 3) Metrica sample-data 준비


In [ ]:
%cd /content/soccer-vit-linebreak
!mkdir -p data/external
if [ ! -d data/external/sample-data ]; then git clone https://github.com/metrica-sports/sample-data data/external/sample-data; fi
!find data/external/sample-data -maxdepth 3 -type d | head -n 20


## 4) Dataset build (한 번만)


In [ ]:
%cd /content/soccer-vit-linebreak
!PYTHONPATH=src python -m soccer_vit.train build-dataset --config configs/default.yaml
!cat data/processed/dataset_summary.json


## 5) Baseline + ViT 스모크 (GPU)

`configs/vit_mid.yaml`는 CPU-safe로 설정돼 있으므로, Colab에서는 아래 셀에서 GPU용으로 덮어써서 실행합니다.


In [ ]:
%cd /content/soccer-vit-linebreak
import yaml
from pathlib import Path
cfg = yaml.safe_load(Path('configs/vit_mid.yaml').read_text())
cfg['train']['device'] = 'cuda'
cfg['train']['batch_size'] = 16
cfg['train']['epochs_stage_a'] = 4
cfg['train']['epochs_stage_b'] = 2
cfg['train']['max_train_samples'] = 1024
cfg['train']['max_val_samples'] = 256
cfg['train']['max_test_samples'] = 256
cfg['eval']['max_eval_samples'] = 256
cfg['eval']['max_counterfactual_eval_samples'] = 128
Path('configs/vit_colab_gpu.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print('wrote configs/vit_colab_gpu.yaml')


In [ ]:
%cd /content/soccer-vit-linebreak
!PYTHONPATH=src python -m soccer_vit.train fit --model baseline_rule_like --config configs/default.yaml
!PYTHONPATH=src python -m soccer_vit.train fit --model baseline_strict --config configs/default.yaml
!PYTHONPATH=src python -m soccer_vit.train fit --model vit_base --config configs/vit_colab_gpu.yaml


## 6) Eval / Report / Explainability 확인


In [ ]:
%cd /content/soccer-vit-linebreak
!MPLBACKEND=Agg PYTHONPATH=src python -m soccer_vit.eval run --config configs/vit_colab_gpu.yaml
!MPLBACKEND=Agg PYTHONPATH=src python -m soccer_vit.report make --config configs/vit_colab_gpu.yaml --n-samples 8
!cat reports/vit_mid/metrics.json | head -n 80


In [ ]:
from pathlib import Path
import json
m = json.loads(Path('reports/vit_mid/metrics.json').read_text())
print('ViT AUROC:', m['models']['vit_base']['original']['auroc'])
print('Counterfactual ViT:', m.get('counterfactual', {}).get('vit_base'))
print('Rollout focus:', m.get('explainability', {}).get('rollout_focus'))


## 7) Figure 미리보기


In [ ]:
from IPython.display import Image, display
fig_dir = Path('reports/vit_mid/figures')
for p in sorted(fig_dir.glob('*rollout.png'))[:3]:
    print(p)
    display(Image(filename=str(p)))
